---
#**Instalação**
---

In [ ]:
!pip install pyarrow matplotlib seaborn ipywidgets -q


In [ ]:
#import zipfile   # [REMOVIDO] só era usado para abrir os .zip
#import io        # [REMOVIDO] só era usado para ler o conteúdo do .zip
import requests
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm
import json
from sklearn.preprocessing import MinMaxScaler
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from IPython.display import display
import numpy as np

---
#**Configurações**
---

In [ ]:
TABLE_SCR  = "scr_data"
TABLE_PIX  = "pix_transacoes"
TABLES_IBGE = ["taxa_escolarizacao", "censo_demografico", "populacao_por_idade"]

PASTA = Path("/content/dados")
PASTA.mkdir(parents=True, exist_ok=True)

#Rodar com o localtunnel
# [REMOVIDO] (GJ00-60) URL_API_LOCAL: o túnel antigo não existe mais. O link novo fica na célula
# "Enviar resultados para o banco", no fim da execução principal.
HEADERS_BCB = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://olinda.bcb.gov.br/",
    "bypass-tunnel-reminder": "true"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS_BCB)

MAX_GET_RETRIES = 5
# [REMOVIDO] Constantes da leitura por .zip/.csv: CHUNK_SIZE, USECOLS_SCR, ano_anterior, URLS_SCR e SCR_LIMITE_DADOS
# (a leitura do SCR agora usa RENAME_SCR e URL_SCR_MES, definidos mais abaixo)

# [REMOVIDO] Colunas do CSV antigo (a API usa o RENAME_SCR)
# USECOLS_SCR = ["data_base", "uf", "porte", "cliente", "modalidade",
#     "carteira_inadimplencia", "carteira_vencida", "carteira_ativa", "vencido_15_a_60_dias", "vencido_61_a_90_dias", "vencido_acima_de_90_dias"]

# # [REMOVIDO] Anos e links dos .zip do BCB (a API usa o URL_SCR_MES)
# ano_anterior = datetime.now().year - 1
# URLS_SCR = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_anterior+1)]

# [REMOVIDO] Limite de linhas do CSV antigo
# SCR_LIMITE_DADOS = 100000
# ano_anterior = datetime.now().year - 1
# URLS_SCR = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_anterior+1)]

####ADICIONADO LUAN - Configurações da API do SCR por sub-região####

# [NOVO] Endereço base da API: tudo que vem antes do nome do endpoint
URL_BASE_SCR = "https://olinda.bcb.gov.br/olinda/servico/scr_sub_regiao/versao/v1/odata"

# [NOVO] Nome do endpoint (o recurso que vamos consultar)
ENDPOINT_SCR = "scr_sub_regiao"

# [NOVO] Tradutor: nome do campo na API -> nome da coluna que o notebook já usa
RENAME_SCR = {
    "DATA_BASE": "data_base",
    "CLIENTE": "cliente",
    "ESTADO": "uf",
    "SUB_REGIAO": "sub_regiao",                       # sub-região mantida conforme a task manda
    "MODALIDADE": "modalidade",
    "RISCO": "classe",                                # ATENÇÃO: substitui o antigo "porte"
    "CARTEIRA": "carteira_ativa",
    "VENCIDO_ACIMA_DE_15_DIAS": "carteira_vencida",   # ATENÇÃO: confirmar o significado
}

# [NOVO] Lista de campos que serão pedidos à API (parâmetro $select), vinda das chaves acima
CAMPOS_API_SCR = ",".join(RENAME_SCR.keys())

# [NOVO] Molde da URL de um mês. O <ANO_MES> é trocado depois por 202401, 202402...
URL_SCR_MES = (f"{URL_BASE_SCR}/{ENDPOINT_SCR}(DataBase=@DataBase)"
               f"?@DataBase=<ANO_MES>&$format=json&$select={CAMPOS_API_SCR}")

# [NOVO] Quantos meses para trás comparar no crescimento da carteira vencida (agingDivida)
# 12 = mesmo mês do ano anterior (tira a sazonalidade); 1 = mês anterior
PERIODOS_CRESCIMENTO = 12

# [REMOVIDO] NIVEIS_SALARIO (mapa de salário/porte para classe A a E): a API traz RISCO (AA-C, D-H, N/D) e não existe mais a coluna "porte"
# NIVEIS_SALARIO = {
#     # Salário Pessoa Física (PF)
#     "Acima de 20 salários mínimos": "A",
#     "Mais de 10 a 20 salários mínimos": "A",
#     "Mais de 5 a 10 salários mínimos": "B",
#     "Mais de 3 a 5 salários mínimos": "C",
#     "Mais de 2 a 3 salários mínimos": "D",
#     "Mais de 1 a 2 salários mínimos": "D",
#     "Até 1 salário mínimo": "E",
#     "Sem rendimento": "E",
#     "a": "A",

#     # Salário Pessoa Jurídica (PJ)
#     "Grande": "A",
#     "Médio": "B",
#     "Pequeno": "C",
#     "Micro": "D",
# }

PIX_LIMITE_DADOS = 35000
# [ALTERADO] Adicionado "&$orderby=AnoMes": sem isso a API devolvia as linhas em ordem qualquer e o $top cortava a maior parte do mês pedido
URL_PIX_API = f"https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='<ANO_MES>'&$top={PIX_LIMITE_DADOS}&$orderby=AnoMes&$format=json&$select=AnoMes,Estado_Ibge,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ"
LINKS_IBGE = {
    "taxa_escolarizacao": {
      "url": "https://apisidra.ibge.gov.br/values/t/7138/n3/all/v/10276/p/last%203/c2/6794/c58/100052,108866/d/v10276%201?formato=json",
      "use_cols": ["V", "D1N", "D3N"]
    },
    "censo_demografico": {
        "url": "https://apisidra.ibge.gov.br/values/t/4709/n1/all/n2/all/n3/all/v/all/p/all/d/v10605%202/l/p,v,t?formato=json",
        "use_cols": ["V" ,"D1N", "D2N", "D3N"]
    },
    "populacao_por_idade": {
        "url": "https://apisidra.ibge.gov.br/values/t/9514/n3/all/v/allxp/p/all/c2/6794/c287/6653,49108,49109,60040,60041,93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93095,93096,93097/c286/113635?formato=json",
        "use_cols": ["V", "D1N", "D3N", "D5N"]
    }
}

map_idade_media = {
    "0 a 4 anos": 2.0,
    "5 a 9 anos": 7.0,
    "10 a 14 anos": 12.0,
    "15 a 19 anos": 17.0,
    "20 a 24 anos": 22.0,
    "25 a 29 anos": 27.0,
    "30 a 34 anos": 32.0,
    "35 a 39 anos": 37.0,
    "40 a 44 anos": 42.0,
    "45 a 49 anos": 47.0,
    "50 a 54 anos": 52.0,
    "55 a 59 anos": 57.0, # [NOVO] Faixa etária adicionada para completar as 19 faixas do IBGE e evitar valores nulos
    "60 a 64 anos": 62.0,
    "65 a 69 anos": 67.0,
    "70 a 74 anos": 72.0,
    "75 a 79 anos": 77.0, # [NOVO] Faixa etária adicionada para completar as 19 faixas do IBGE e evitar valores nulos
    "80 a 84 anos": 82.0,
    "85 a 89 anos": 87.0,
    "90 a 94 anos": 92.0,
    "95 a 99 anos": 97.0,
    "100 anos ou mais": 100.0
}

---
#**Funções de Leitura**
---

In [ ]:
import time

def baixar_com_retry(url, SESSION, TIMEOUT):  #Leitura geral de arquivos
  primeria_tentativa = True
  headers_req = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
      "Accept": "application/json, text/plain, */*",
      "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
      "Referer": "https://olinda.bcb.gov.br/",
  }
  for i in range(1, MAX_GET_RETRIES + 1): #Faz get com limite de tentativas
    try:
      resposta = SESSION.get(url, headers=headers_req, timeout=TIMEOUT) #Tenta em um tempo curto
      resposta.raise_for_status()
      resposta.encoding = "utf-8-sig"
      return resposta
    except Exception as e:
      if i < MAX_GET_RETRIES:
        time.sleep(0.5 * i)
      else:
        print(f"Todas as {MAX_GET_RETRIES} tentativas falharam: {e}")
        return None


In [ ]:
def ler_json(url, SESSION, timeout = 5, chave = None, funcao_tratamento = None, filtro_colunas = None):
  resposta = baixar_com_retry(url, SESSION, timeout)
  if resposta is None:
    return

  json = resposta.json()

  if(chave and chave in json):
    df = pd.DataFrame(json[chave])
  else:
    df = pd.DataFrame(json)

  if filtro_colunas:
    df = df[filtro_colunas]
  if funcao_tratamento:
    return funcao_tratamento(df)

  return df

---
#**Função para enviar dados**
---

In [ ]:
# [ALTERADO] (GJ00-60) Envio dos resultados para a API Flask, que grava no banco (POST /api/carga).
# Antes: uma requisição por linha, com os cabeçalhos do BCB. Agora: uma requisição por mês, com um token.
# O formato é o mesmo que a API devolve ao site (frontend/dados/mock_sub_regioes.json).
CAMPOS_CHAVE = ["ano_mes", "regiao", "uf", "sub_regiao"]
CAMPOS_EIXO_I = ["inadimplenciaReal", "fragilidadeRenda", "agingDivida", "vulnerabilidadeSocial"]
CAMPOS_EIXO_II = ["maturidadePix", "crescimentoPopulacional", "totalHabitantes", "bonusDemografico"]
CAMPOS_SCORE = ["score", "faixa", "ranking"]


def _valor_json(valor):
    """Vazio (NaN) vira null e números do numpy viram números comuns: o JSON não aceita NaN."""
    if valor is None or (not isinstance(valor, str) and pd.isna(valor)):
        return None
    if isinstance(valor, np.integer):
        return int(valor)
    if isinstance(valor, (np.floating, float)):
        return round(float(valor), 2)
    return valor


def montar_lotes(df_score):
    """Um lote por mês no formato do contrato: {"eixo_i": [...], "eixo_ii": [...]}, com score, faixa e ranking."""
    lotes = {}
    for mes, grupo in df_score.groupby("ano_mes", sort=True):
        # a API recusa o mês inteiro se uma linha tiver chave inválida; essas linhas ficam fora e o aviso mostra quantas
        chave_ok = (grupo["uf"].astype(str).str.fullmatch(r"[A-Z]{2}")
                    & grupo["sub_regiao"].astype(str).str.fullmatch(r"\d{2}"))
        if not chave_ok.all():
            print(f"{mes}: {int((~chave_ok).sum())} linha(s) sem UF ou sub-região válida ficaram fora do envio")
        linhas = grupo[chave_ok].to_dict("records")
        lotes[str(mes)] = {
            eixo: [{c: _valor_json(linha[c]) for c in CAMPOS_CHAVE + campos + CAMPOS_SCORE} for linha in linhas]
            for eixo, campos in (("eixo_i", CAMPOS_EIXO_I), ("eixo_ii", CAMPOS_EIXO_II))
        }
    return lotes


def enviar_resultados(df_score, url_api, token):
    """Manda mês a mês para o Flask (ligado no PC, com o túnel). Para no primeiro erro, sem esconder o motivo."""
    if not url_api or not token:
        raise ValueError("Preencha URL_API_LOCAL com o link do túnel e crie o secret CARGA_TOKEN no Colab")
    total = 0
    for mes, lote in montar_lotes(df_score).items():
        resposta = requests.post(url_api.rstrip("/") + "/api/carga", json=lote,
                                 headers={"X-Token-Carga": token}, timeout=60)
        if resposta.status_code != 201:
            raise RuntimeError(f"Mês {mes}: erro {resposta.status_code} - {resposta.text[:300]}")
        gravadas = resposta.json()["linhas_gravadas"]
        total += gravadas
        print(f"{mes}: {gravadas} linhas gravadas")
    print(f"Total: {total} linhas gravadas no banco")
    return total


---
#**Leitura dados SCR**
---

In [ ]:
# [ALTERADO] Antes baixava um .zip e lia o CSV em pedaços. Agora faz uma chamada JSON por mês
def processar_url_scr(args):
    url, SESSION = args
    ano_mes = url.split("@DataBase=")[1][:6]     # [NOVO] só para mostrar em qual mês deu erro
    try:
        # [ALTERADO] usa o ler_json (que já usa o baixar_com_retry) no lugar do zipfile/read_csv
        return ler_json(url=url, SESSION=SESSION, timeout=30, chave="value")
    except Exception as e:
        print(f"SCR {ano_mes} falhou: {e}")      # [NOVO] um mês com problema não derruba os outros
        return None

def ler_scr(SESSION):
    # [ALTERADO] Antes: lista de anos (.zip). Agora: lista de meses (AAAAMM) desde 2021-01
    anos_meses = gerar_anos_meses_filtrado("2021-01-01")

    # [ALTERADO] Antes: URLs dos zips. Agora: a URL da API de cada mês
    urls = [URL_SCR_MES.replace("<ANO_MES>", ano_mes) for ano_mes in anos_meses]

    args = [(url, SESSION) for url in urls]       # [IGUAL]

    # [ALTERADO] max_workers=4: poucas chamadas ao mesmo tempo, para não sobrecarregar a API do BCB
    with ThreadPoolExecutor(max_workers=4) as executor:
        resultados = list(tqdm(executor.map(processar_url_scr, args), total=len(urls), desc="Baixando SCR..."))

    # [ALTERADO] Cada resultado agora é um DataFrame (ou None se o mês falhou), não uma lista deles
    dfs = [r for r in resultados if r is not None and not r.empty]
    if not dfs:                                   # [IGUAL]
        print("Nenhum dado do SCR encontrado.")
        return

    df = pd.concat(dfs, ignore_index=True)        # [IGUAL] junta todos os meses
    df = tratamento_scr_api(df)                   # [ALTERADO] trata uma vez só, depois de juntar
    print(f"Lido SCR: {df.shape[0]} linhas, {df.shape[1]} colunas")   # [IGUAL]
    salvar_parquet(df, TABLE_SCR)                 # [IGUAL] salva em scr_data.parquet

---
#**Leitura dados Pix**
---

In [ ]:
####ALTERADO LUAN - Agora aceita uma data de início (o Pix continua igual, pois o padrão é 2020-11-01)####

def gerar_anos_meses_filtrado(inicio="2020-11-01"):     # [ALTERADO] ganhou o parâmetro "inicio"
    data_atual = datetime.now()
    anos_meses = pd.date_range(start=inicio, end=data_atual, freq="MS")   # [ALTERADO] usa "inicio" no lugar da data fixa
    lista_completa = anos_meses[:-1].strftime("%Y%m").tolist()            # [IGUAL] tira o mês atual e formata como "202101"

    return lista_completa

In [ ]:
def processar_url_pix(args):
    url, SESSION = args
    # [NOVO] Mês que esta chamada pediu (vem da própria URL: @DataBase='202609')
    ano_mes = url.split("@DataBase='")[1][:6]

    # [ALTERADO] Antes: return ler_json(...) devolvia tudo que a API mandava (mês pedido + seguintes)
    df = ler_json(url=url, SESSION=SESSION, timeout=10, chave="value", funcao_tratamento=tratamento_pix)
    if df is None:                                     # [NOVO] segurança, igual ao resto do notebook
        return None

    # [NOVO] Fica só com o mês pedido, descartando os meses extras que a API manda junto
    return df[df["AnoMes"].astype(str) == ano_mes]
def ler_pix(SESSION):
    anos_meses = gerar_anos_meses_filtrado()

    URLS_PIX = [URL_PIX_API.replace("<ANO_MES>", ano_mes) for ano_mes in anos_meses]

    args = [(url, SESSION) for url in URLS_PIX]
    with ThreadPoolExecutor(max_workers=4) as executor:   # [ALTERADO] antes: ThreadPoolExecutor()
        resultado = list(tqdm(executor.map(processar_url_pix, args), total=len(URLS_PIX), desc="Baixando arquivos pix..."))

    if not any(r is not None and not r.empty for r in resultado):
        print("Nenhum dado do PIX encontrado.")
        return

    df = pd.concat([r for r in resultado if r is not None and not r.empty], ignore_index=True)
    print(f"Lido PIX: {df.shape[0]} linhas, {df.shape[1]} colunas")
    salvar_parquet(df, TABLE_PIX)

---
#**Leitura dados IBGE**
---

In [ ]:
def ler_urls_ibge(SESSION):
    funcoes_tratamento_ibge = [tratamento_taxa_escolarizacao, tratamento_censo_demografico, tratamento_populacao_por_idade]
    tratamentos = {k: v for k, v in zip(TABLES_IBGE, funcoes_tratamento_ibge)}

    for table_name, item in LINKS_IBGE.items():
        print(f"Baixando {table_name}...")
        df = ler_json(item["url"], SESSION=SESSION, timeout=10, filtro_colunas=item["use_cols"], funcao_tratamento=tratamentos[table_name])

        if df is not None and not df.empty:
            salvar_parquet(df, table_name)
        else:
            print(f"Nenhum dado encontrado para {table_name}.")

---
#**Tratamento**
---

In [ ]:
#Funções gerais de tratamento de dados

def limpeza_string(df, colunas=None):
    if colunas is None:
        colunas = df.select_dtypes(include="object").columns.tolist()
    else:
        colunas = [col for col in colunas if col in df.columns and df[col].dtype == "object"]

    if not colunas:
        return df

    for col in colunas:
        df[col] = df[col].str.strip()
    return df

def converter_numerico(df, *colunas):
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def limpar_nulos(df):
    numericas = df.select_dtypes(include="number").columns
    df[numericas] = df[numericas].fillna(df[numericas].median())

    categoricas = df.select_dtypes(include="object").columns
    df[categoricas] = df[categoricas].fillna("NAO_INFORMADO")
    return df

def formatar_data(df, colunas, formato="%Y-%m-%d"):
    if not colunas: return df
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_datetime(df[col], format=formato).dt.strftime("%Y%m")
    return df

def formatar_numero(df, colunas):
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_numeric(df[col].str.replace(",", "."), errors="coerce")
    return df

# [REMOVIDO] def converter_nivel_salario(df, coluna): ... (dependia do NIVEIS_SALARIO; não é mais chamada)
# def converter_nivel_salario(df, coluna):
#   df[coluna] = df[coluna].map(NIVEIS_SALARIO)
#   return df

def converter_uf(df, coluna):
  estados_brasil = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF", "Espírito Santo": "ES",
    "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT", "Mato Grosso do Sul": "MS",
    "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC",
    "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO",
    11: "RO", 12: "AC", 13: "AM", 14: "RR",
    15: "PA", 16: "AP", 17: "TO", 21: "MA",
    22: "PI", 23: "CE", 24: "RN", 25: "PB",
    26: "PE", 27: "AL", 28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS", 50: "MS",
    51: "MT", 52: "GO", 53: "DF"
  }

  df = df.rename(columns = {
      coluna: "uf"
  })
  df["uf"] = df["uf"].map(estados_brasil)

  return df

def converter_uf_para_macro_regiao(df, coluna):
  mapeamento_regioes = {
      'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
      'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
      'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
      'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
      'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
      'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
  }

  df['regiao'] = df[coluna].map(mapeamento_regioes)

  return df

def converter_porcentagem(df, coluna):
  df[coluna] = pd.to_numeric(df[coluna], errors='coerce') / 100
  return df

In [ ]:
## [REMOVIDO] def tratamento_scr(df): ... return df   (era o tratamento do CSV, trocado pelo tratamento_scr_api)

####ADICIONADO LUAN - Tratamento dos dados vindos da API (diferente do que vinha do CSV)####

def tratamento_scr_api(df):
    # [NOVO] Troca os nomes da API pelos nomes do notebook (usa o RENAME_SCR das Configurações)
    # .copy() evita avisos do pandas ao alterar o DataFrame depois
    df = df.rename(columns=RENAME_SCR).copy()

    # [ALTERADO] Mesma limpeza de espaços de antes, com as colunas de texto da API
    df = limpeza_string(df, ["cliente", "uf", "sub_regiao", "classe", "modalidade"])

    # [NOVO] A API manda classe nula (None) em vários meses; vira "N/D" (igual ao que a API já usa)
    # Sem isso, o groupby do padronizar_df_scr descartaria essas linhas e a carteira somaria menos
    df["classe"] = df["classe"].fillna("N/D")

    # [ALTERADO] A API manda 202401 como número inteiro; aqui vira texto "202401"
    # Antes: formatar_data convertia "2024-01-01" em "202401" (daria erro com os dados da API)
    df["data_base"] = df["data_base"].astype(str)

    # [NOVO] "NI" = estado não identificado. Não existe no mapa de regiões e viraria regiao = NaN
    df = df[df["uf"] != "NI"].copy()

    # [IGUAL] Cria a coluna "regiao" (Norte, Sul...) a partir da uf
    df = converter_uf_para_macro_regiao(df, "uf")

    # [ALTERADO] Antes: formatar_numero (usava .str.replace, que só funciona com texto)
    # Agora os valores já chegam como float, então só garantimos o tipo numérico
    df = converter_numerico(df, "carteira_ativa", "carteira_vencida")

    # [IGUAL] Valor ausente vira R$ 0,00
    df[["carteira_ativa", "carteira_vencida"]] = df[["carteira_ativa", "carteira_vencida"]].fillna(0.0)

    # [REMOVIDO] limpar_nulos, converter_uf e converter_nivel_salario:
    #   a API já traz a sigla da UF e não existe mais a coluna "porte"
    return df

In [ ]:
def tratamento_pix(df):    # [ALTERADO] estava comentada com "#"; o ler_pix precisa dela ativa
  df = limpar_nulos(df)    # [IGUAL] daqui para baixo nada muda
  df = limpeza_string(df, ["AnoMes"])
  df = converter_numerico(df, "VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ")
  df = converter_uf(df, "Estado_Ibge")
  df = converter_uf_para_macro_regiao(df, "uf")
  return df

In [ ]:
def tratamento_taxa_escolarizacao(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = converter_porcentagem(df, "V")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_censo_demografico(df):
  df = limpar_nulos(df)
  df["V"] = df["V"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
  df = converter_numerico(df, "V")
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_populacao_por_idade(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N", "D5N"])
  df = converter_numerico(df, "V")
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor", "D5N": "idade"})
  df["idade"] = df["idade"].map(map_idade_media)
  return df

---
#**Perpetuação**
---

In [ ]:
def salvar_parquet(df, nome):
  caminho = PASTA / f"{nome}.parquet"
  df.to_parquet(caminho, index=False)
  print(f" --Salvo em: {caminho}")

# **Refatoração**

In [ ]:
def validar_consistencia_saldos(df):
    """Função para checar se a soma dos saldos por modalidade é igual à carteira ativa total."""
    # 1. Filtra registros de carteira ativa
    df_ativa = df[df["metrica"] == "carteira_ativa"]

    # 2. Agrupa por modalidade e calcula a soma total
    soma_modalidades = (
        df_ativa.groupby(["ano_mes", "uf", "modalidade"])["valor"]
        .sum()
        .groupby(["ano_mes", "uf"])
        .sum()
        .reset_index(name="soma_modalidades")
    )

    # 3. Obtém o total da carteira ativa informado para o período/UF
    total_carteira = (
        df_ativa.groupby(["ano_mes", "uf"])["valor"]
        .sum()
        .reset_index(name="total_carteira_ativa")
    )

    # 4. Cruza e valida a diferença
    checagem = pd.merge(soma_modalidades, total_carteira, on=["ano_mes", "uf"])
    checagem["diferenca"] = (
        checagem["soma_modalidades"] - checagem["total_carteira_ativa"]
    ).round(2)

    inconsistencias = checagem[checagem["diferenca"] != 0]

    if not inconsistencias.empty:
        print(
            f"[VALIDAÇÃO] Encontradas {len(inconsistencias)} inconsistências entre modalidades e carteira ativa!"
        )
    else:
        print(
            "[VALIDAÇÃO] Saldos entre modalidades e carteira ativa estão 100% consistentes!"
        )

def padronizar_df_scr(df_scr):
    df_scr = df_scr.rename(columns={"data_base": "ano_mes"})                      # [IGUAL]

    df_scr["tipo"] = df_scr["cliente"].map({"PF": "pf"}).fillna("pj")             # [IGUAL]

    df_scr = df_scr.melt(
        id_vars=["ano_mes", "regiao", "uf", "sub_regiao", "tipo", "classe", "modalidade"],   # [ALTERADO] + "sub_regiao"
        value_vars=["carteira_vencida", "carteira_ativa"],                        # [IGUAL]
        var_name="metrica",
        value_name="valor",
    )
    df_scr["origem"] = "scr"                                                      # [IGUAL]

     # [REMOVIDO] validar_consistencia_saldos(df_scr): comparava a soma do dado com ela mesma e rodava por chunk (mensagem repetida)

    for col in ["tipo", "classe", "origem", "metrica", "uf", "sub_regiao", "modalidade"]:   # [ALTERADO] + "sub_regiao"
        if col in df_scr.columns:
            df_scr[col] = df_scr[col].astype("category")

    df_scr = (
        df_scr.groupby(
            [
                "ano_mes",
                "regiao",
                "uf",
                "sub_regiao",                                                     # [NOVO]
                "tipo",
                "classe",
                "modalidade",
                "metrica",
                "origem",
            ],
            observed=True,
        )["valor"]
        .sum()
        .reset_index()
    )

    return df_scr[[
        "ano_mes",
        "regiao",
        "uf",
        "sub_regiao",                                                             # [NOVO]
        "tipo",
        "classe",
        "metrica",
        "origem",
        "valor",
    ]]

In [ ]:
def validar_scr_contra_origem(caminho_origem="/content/dados/scr_data.parquet",
                              caminho_final="/content/dados/scr_pix.parquet"):   # [NOVO]
    origem = pd.read_parquet(caminho_origem)
    origem["data_base"] = origem["data_base"].astype(str)
    esperado = origem.groupby("data_base")[["carteira_ativa", "carteira_vencida"]].sum()

    final = pd.read_parquet(caminho_final)
    final = final[final["origem"] == "scr"]
    obtido = final.groupby(["ano_mes", "metrica"], observed=True)["valor"].sum().unstack()

    dif = (esperado - obtido[["carteira_ativa", "carteira_vencida"]]).abs()
    erros = dif[(dif > 1).any(axis=1)]

    if erros.empty:
        print(f"[VALIDAÇÃO] SCR: soma da origem = soma do scr_pix em {len(esperado)} meses.")
    else:
        print(f"[VALIDAÇÃO] SCR: divergência em {len(erros)} meses (diferença em R$):")
        print(erros)

In [ ]:
# [REMOVIDO] def padronizar_df_pix(df_pix): ... (cópia antiga da seção Refatoração)
# Ela gerava as colunas extras "modalidade" e "categoria_credito", mas era sobrescrita
# pela versão da seção Scoring, que é a que de fato roda. Fica só a versão da seção Scoring.
# import numpy as np
# import pandas as pd


# def padronizar_df_pix(df_pix):
#     df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

#     df_pix = df_pix.melt(
#         id_vars=["ano_mes", "regiao", "uf"],
#         value_vars=[
#             "VL_PagadorPF",
#             "QT_PagadorPF",
#             "VL_PagadorPJ",
#             "QT_PagadorPJ",
#             "QT_PES_PagadorPF",
#             "QT_PES_PagadorPJ",
#         ],
#         var_name="metrica_raw",
#         value_name="valor",
#     )

#     df_pix["tipo"] = (
#         df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})
#     )

#     df_pix["metrica"] = np.select(
#         [
#             df_pix["metrica_raw"].str.contains("VL"),
#             df_pix["metrica_raw"].str.contains("QT_PES"),
#             df_pix["metrica_raw"].str.contains("QT"),
#         ],
#         ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
#         default="outros",
#     )

#     df_pix["origem"] = "pix"

#     df_pix = (
#         df_pix.groupby(
#             ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
#             observed=True,
#         )["valor"]
#         .sum()
#         .reset_index()
#     )

#     # Compatibilidade de colunas com o schema do SCR
#     df_pix["classe"] = pd.NA
#     df_pix["modalidade"] = "pix"
#     df_pix["categoria_credito"] = "nao_aplica"

#     for col in [
#         "tipo",
#         "origem",
#         "metrica",
#         "uf",
#         "regiao",
#         "classe",
#         "modalidade",
#         "categoria_credito",
#     ]:
#         if col in df_pix.columns:
#             df_pix[col] = df_pix[col].astype("category")

#     return df_pix[[
#         "ano_mes",
#         "regiao",
#         "uf",
#         "tipo",
#         "classe",
#         "modalidade",
#         "categoria_credito",
#         "metrica",
#         "origem",
#         "valor",
#     ]]

---
#**Pré Processamento**
---

In [ ]:
def processar():
    print(f"--- Iniciando Extração de Dados ---")

    SESSION = requests.Session()
    SESSION.headers.update(HEADERS_BCB)

    try:
        ler_scr(SESSION)
    except Exception as e:
        print(f"Erro ao ler SCR: {e}")

    try:
        ler_pix(SESSION)
    except Exception as e:
        print(f"Erro ao ler Pix: {e}")

    try:
        ler_urls_ibge(SESSION)
    except Exception as e:
        print(f"Erro ao ler IBGE: {e}")

---
#**Scoring**
---

In [ ]:
def extract_data(path:str):
    df = pd.read_parquet(path)
    return df

Leitura Parquets Pix e Scr
---

In [ ]:
# [REMOVIDO] def padronizar_df_scr(df_scr): ... (cópia antiga da seção Scoring)
# Fica só a versão da seção Refatoração, já ajustada para as colunas da API

In [ ]:
def padronizar_df_pix(df_pix):
    df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

    df_pix = df_pix.melt(
        id_vars=["ano_mes", "regiao", "uf"],
        value_vars=["VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ"],
        var_name="metrica_raw",
        value_name="valor"
    )

    df_pix["tipo"] = df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})

    df_pix["metrica"] = np.select(
        [df_pix["metrica_raw"].str.contains("VL"),
         df_pix["metrica_raw"].str.contains("QT_PES"),
         df_pix["metrica_raw"].str.contains("QT")],
        ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
        default="outros"
    )

    df_pix["origem"] = "pix"

    df_pix = df_pix.groupby(
        ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
        observed=True
    )["valor"].sum().reset_index()

    df_pix["classe"] = pd.NA  # adiciona depois do groupby

    for col in ["tipo", "origem", "metrica", "uf", "regiao"]:
        df_pix[col] = df_pix[col].astype("category")

    return df_pix[["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem", "valor"]]

In [ ]:
def construir_df_scr_pix():
  if not Path('/content/dados/pix_transacoes.parquet').exists() or not Path('/content/dados/scr_data.parquet').exists():
      raise FileNotFoundError("Arquivos pix_transacoes.parquet ou scr_data.parquet não foram gerados pela API do BCB. Verifique a conexão com a API do Banco Central na etapa anterior.")

  chunks_pix = []
  chunks_scr = []

  #Padroniza o parquet pix por chunk
  tabela_pix = pq.ParquetFile('/content/dados/pix_transacoes.parquet')

  for batch in tabela_pix.iter_batches(batch_size=100000):
    chunks_pix.append(padronizar_df_pix(batch.to_pandas()))
  df_pix = pd.concat(chunks_pix, ignore_index=True)

  df_pix = df_pix.groupby(                                                   # [IGUAL] groupby do Pix (sem sub_regiao)
      ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
      observed=True
  )["valor"].sum().reset_index()
  df_pix["classe"] = pd.NA                                                   # [IGUAL]
  df_pix["sub_regiao"] = pd.NA                                               # [NOVO]

  #Padroniza o parquet scr por chunk
  tabela_scr = pq.ParquetFile('/content/dados/scr_data.parquet')

  for batch in tabela_scr.iter_batches(batch_size=100000):
    chunks_scr.append(padronizar_df_scr(batch.to_pandas()))
  df_scr = pd.concat(chunks_scr, ignore_index=True)

  df_scr = df_scr.groupby(
      ["ano_mes", "regiao", "uf", "sub_regiao", "tipo", "classe", "metrica", "origem"],   # [ALTERADO] + "sub_regiao"
      observed=True
  )["valor"].sum().reset_index()

  df_pix["ano_mes"] = df_pix["ano_mes"].astype(str)                          # [IGUAL]
  df_scr["ano_mes"] = df_scr["ano_mes"].astype(str)                          # [IGUAL]

  #une os dois parquets em um individual
  df_scr_pix = pd.concat([df_scr, df_pix], ignore_index=True)                # [IGUAL]

  salvar_parquet(df_scr_pix, "scr_pix")                                      # [IGUAL]

Leitura Parquets IBGE
---

In [ ]:
def calculo_idade():
  #---populacao por idade
  df_idade = pd.read_parquet('/content/dados/populacao_por_idade.parquet')

  df_idade["idade_ponderada"] = df_idade["idade"] * df_idade["valor"]

  df_idade = (
      df_idade.groupby(["ano", "regiao", "uf"])
      .agg(
          soma_idade=("idade_ponderada", "sum"),
          soma_pop=("valor", "sum")
      )
      .reset_index()
  )

  df_idade["idade_media"] = df_idade["soma_idade"] / df_idade["soma_pop"]
  # [ALTERADO] Removido o cálculo de diferença (idade_max - idade_media);
  # Agora atribue a idade media direto ao indicador, mantendo o nome da coluna por compatibilidade
  df_idade["bonusDemografico"] = df_idade["idade_media"]

  return df_idade[["ano", "regiao", "uf", "bonusDemografico"]]

In [ ]:
def construir_df_ibge():
  #---escolarizacao
  df_escolarizacao = extract_data('/content/dados/taxa_escolarizacao.parquet')

  df_escolarizacao = df_escolarizacao.groupby(  #realiza a média de valores com mesmo ano, regiao e uf
      ["ano", "regiao", "uf"],
      observed=True)["valor"].mean().reset_index()
  df_escolarizacao = df_escolarizacao.rename(columns={"valor": "taxa_escolarizacao"})

  #---censo demografico
  df_censo_demografico = extract_data('/content/dados/censo_demografico.parquet')

  #separa colunas por metrica
  df_censo_demografico = df_censo_demografico.pivot_table(
      index=["ano", "regiao", "uf"],
      columns="D2N",
      values="valor").reset_index()

  df_censo_demografico = df_censo_demografico.rename(columns={
      "População residente": "populacao_residente",
      "Taxa de crescimento geométrico": "taxa_crescimento",
      "Variação absoluta da população residente 2010 compatibilizada": "variacao_populacao"
  })

  df_idade = calculo_idade()

  #arruma porcentagem
  df_censo_demografico["taxa_crescimento"] = df_censo_demografico["taxa_crescimento"]/100

  df_censo_demografico.columns.name = None

  # [NOVO] Censo e idade só existem em anos de censo (2010, 2022), então não casam com os anos da escolarização.
  # Fica com o dado mais recente de cada UF (.last()) e tira a coluna "ano", para valer para todos os anos
  df_censo_demografico = (df_censo_demografico.sort_values("ano")
      .groupby(["regiao", "uf"]).last().reset_index().drop(columns=["ano"]))
  df_idade = (df_idade.sort_values("ano")
      .groupby(["regiao", "uf"]).last().reset_index().drop(columns=["ano"]))

  # [ALTERADO] Antes cruzava por ["ano", "regiao", "uf"]. Agora cruza só por ["regiao", "uf"]
  df_ibge = df_escolarizacao.merge(
      df_censo_demografico,
      on=["regiao", "uf"],
      how="left").merge(
      df_idade,
      on=["regiao", "uf"],
      how="left")

  # [ALTERADO] Ordena por UF, região e ano, agrupando por UF antes de aplicar fill/bfill;
  # Isso garante que o preenchimento de nulos use apenas dados do mesmo estado, sem vazar para outras UFs
  df_ibge = df_ibge.sort_values(["uf", "regiao", "ano"]).reset_index(drop=True)
  colunas_preencher = ["taxa_escolarizacao", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]
  df_ibge[colunas_preencher] = df_ibge.groupby("uf")[colunas_preencher].transform(lambda x: x.ffill().bfill())

  salvar_parquet(df_ibge, "ibge")


In [ ]:
def normalizacao(df, coluna):
  df[coluna + "_max"] = df.groupby("ano_mes")[coluna].transform("max")
  df[coluna + "_min"] = df.groupby("ano_mes")[coluna].transform("min")
  df[coluna] = (1 + (df[coluna] - df[coluna + "_min"]) / (df[coluna + "_max"] - df[coluna + "_min"]) * 4).round(2)
  df = df.drop(columns=[coluna + "_max", coluna + "_min"])

  return df

---
#**Eixo I – Risco de Crédito**
---

In [ ]:
filtro = ["ano_mes", "regiao", "uf"]


def agrupar_metrica(df, metrica, filtro_extra=None):
    mask = df["metrica"] == metrica
    if filtro_extra is not None:
        mask &= filtro_extra
    return df[mask].groupby(filtro)["valor"].sum()

In [ ]:
def calcular_crescimento_vencida(df_scr_pix_base):
    base = df_scr_pix_base[df_scr_pix_base["metrica"] == "carteira_vencida"]
    atual = (
        base.groupby(["ano_mes", "regiao", "uf"], observed=True)["valor"]
        .sum()
        .reset_index()
    )
    atual = atual.rename(columns={"valor": "vencida"})

    atual["ano_mes"] = atual["ano_mes"].astype(str)
    atual["uf"] = atual["uf"].astype(str)
    atual["regiao"] = atual["regiao"].astype(str)
    atual["data"] = pd.to_datetime(atual["ano_mes"], format="%Y%m")

    passado = atual[["uf", "data", "vencida"]].rename(
        columns={"vencida": "vencida_anterior"}
    )
    passado["data"] = passado["data"] + pd.DateOffset(
        months=PERIODOS_CRESCIMENTO
    )

    atual = atual.merge(passado, on=["uf", "data"], how="left")

    atual["agingDivida"] = np.where(
        atual["vencida_anterior"] > 0,
        (atual["vencida"] / atual["vencida_anterior"]) - 1,
        0.0,
    )
    return atual[["ano_mes", "regiao", "uf", "agingDivida"]]


def calcular_eixo_i(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    df_crescimento = calcular_crescimento_vencida(df_scr_pix_base)

    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "taxa_escolarizacao"]],
        on=["ano", "regiao", "uf"],
        how="left",
    ).drop(columns=["ano"])
        # [NOVO] Ordena por UF e mês: o ffill/bfill abaixo preenche conforme a ordem das linhas.
    # Sem isso, 2021 e 2022 receberiam a escolarização de 2025 em vez da de 2023
    df_calculos = df_calculos.sort_values(["uf", "ano_mes"])

    df_calculos["taxa_escolarizacao"] = df_calculos.groupby("uf")[
        "taxa_escolarizacao"
    ].transform(lambda x: x.ffill().bfill())

    # --- CORREÇÃO 1: Removida a duplicação do agrupar_metrica da carteira_ativa_classes_de ---   # [IGUAL]
    df_metricas = pd.concat(
        [
            agrupar_metrica(df_calculos, "carteira_vencida").rename(
                "carteira_vencida"
            ),                                                                                    # [IGUAL]
            agrupar_metrica(df_calculos, "carteira_ativa").rename(
                "carteira_ativa"
            ),                                                                                    # [IGUAL]

            # [NOVO] DECISÃO (21/10/2026): a API do SCR traz RISCO de crédito (AA-C, D-H, N/D),
            # e não mais a classe de renda (A a E). Por isso "D-H" (risco alto) passa a valer
            # no lugar de "D" e "E". O nome carteira_ativa_classes_de e o indicador
            # fragilidadeRenda foram mantidos para não quebrar o resto do código.
            agrupar_metrica(
                df_calculos, "carteira_ativa", df_calculos["classe"].isin(["D-H"])   # [ALTERADO] antes: .isin(["D", "E"])
            ).rename("carteira_ativa_classes_de"),
        ],
        axis=1,
    ).reset_index()                                                                               # [IGUAL]
    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "taxa_escolarizacao"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )
      # [NOVO] Remove meses que só existem no Pix (202011, 202012, 202609): sem SCR não há carteira
    # e esses meses apareceriam com zeros falsos
    df_calculos = df_calculos.dropna(subset=["carteira_ativa"])

    df_calculos = df_calculos.merge(df_crescimento, on=filtro, how="left")
    df_calculos["agingDivida"] = df_calculos["agingDivida"].fillna(0.0)

    # --- Divisões com tratamento de divisão por zero ---
    df_calculos["inadimplenciaReal"] = np.where(
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_vencida"] / df_calculos["carteira_ativa"],
        0.0,
    )

    # [NOVO] DECISÃO (21/10/2026): a partir de 202501 a API só envia o risco "N/D" (não há "D-H").
    # Nesses meses todas as UFs ficam com fração 0, a normalizacao gera NaN (0/0) e o fillna(0)
    # final transforma em 0. A escala válida é de 1 a 5, então 0 significa "SEM DADO", e não "risco nulo".
    df_calculos["fragilidadeRenda"] = np.where(                                  # [IGUAL]
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_ativa_classes_de"] / df_calculos["carteira_ativa"],
        0.0,
    )

    # --- CORREÇÃO 2: Sintaxe de vulnerabilidadeSocial corrigida ---
    df_calculos["vulnerabilidadeSocial"] = 1 - df_calculos["taxa_escolarizacao"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "inadimplenciaReal")
    df_calculos = normalizacao(df_calculos, "fragilidadeRenda")
    df_calculos = normalizacao(df_calculos, "agingDivida")
    df_calculos = normalizacao(df_calculos, "vulnerabilidadeSocial")

    # --- Limpeza final ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[[
        "ano_mes",
        "regiao",
        "uf",
        "inadimplenciaReal",
        "fragilidadeRenda",
        "agingDivida",
        "vulnerabilidadeSocial",
    ]]



# **Eixo II – Inclusão e Expansão Demográfica**

---



In [ ]:
def calcular_eixo_ii(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    # --- Extrai ano e merge com IBGE ---
    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]],
        on=["ano", "regiao", "uf"],
        how="left"
    ).drop(columns=["ano"])

    # [NOVO] Ordena por UF e mês antes de preencher nulos para manter a cronologia correta dentro de cada estado
    df_calculos = df_calculos.sort_values(["uf", "ano_mes"])

    def eliminar_duplicata(metrica):
        df_calculos[metrica] = (
            df_calculos.groupby("uf")[metrica]
            .transform(lambda x: x.ffill().bfill())
        )

    eliminar_duplicata("populacao_residente")
    eliminar_duplicata("taxa_crescimento")
    eliminar_duplicata("variacao_populacao")
    eliminar_duplicata("bonusDemografico")

    # --- Agrupamento das métricas Pix ---
    df_metricas = pd.concat([
        agrupar_metrica(df_calculos, "vl_pagador").rename("vl_pagador"),
        agrupar_metrica(df_calculos, "qt_pes_pagador").rename("qt_pes_pagador"),
        agrupar_metrica(df_calculos, "qt_pagador").rename("qt_pagador"),
    ], axis=1).reset_index()

    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )

    # [ALTERADO] Salva as duas métricas brutas em colunas para normalizá-las separadamente
    # 1. Maturidade Uso PIX = qt_pagador / qt_pes_pagador
    df_calculos["maturidade_uso"] = np.where(
        df_calculos["qt_pes_pagador"] > 0,
        df_calculos["qt_pagador"] / df_calculos["qt_pes_pagador"],
        0.0
    )

    # 2. Maturidade Financeira PIX = vl_pagador / qt_pagador
    df_calculos["maturidade_financeira"] = np.where(
        df_calculos["qt_pagador"] > 0,
        df_calculos["vl_pagador"] / df_calculos["qt_pagador"],
        0.0
    )

    # [NOVO] Normaliza cada métrica na escala de 1 a 5 antes de aplicar os pesos,
    # evitando que o ticket médio em reais domine a frequência de uso
    df_calculos = normalizacao(df_calculos, "maturidade_uso")
    df_calculos = normalizacao(df_calculos, "maturidade_financeira")

    # 3. Consolidado Maturidade PIX = (Uso * 0.6) + (Financeira * 0.4)
    df_calculos["maturidadePix"] = (df_calculos["maturidade_uso"] * 0.6) + (df_calculos["maturidade_financeira"] * 0.4)

    # Demais indicadores do Eixo II
    df_calculos["crescimentoPopulacional"] = df_calculos["taxa_crescimento"]
    df_calculos["totalHabitantes"] = df_calculos["populacao_residente"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "crescimentoPopulacional")
    df_calculos = normalizacao(df_calculos, "totalHabitantes")
    df_calculos = normalizacao(df_calculos, "bonusDemografico")

    # --- Limpeza final de segurança ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[["ano_mes", "regiao", "uf", "maturidadePix", "crescimentoPopulacional", "totalHabitantes", "bonusDemografico"]]


# **Score por Sub-região – Oportunidade de Crédito**
---
Score de 0 a 100 por UF + sub-região e mês, com faixa e posição no ranking do mês.

In [ ]:
# [NOVO] Célula inteira nova: função do score por sub-região.
# SCORE POR SUB-REGIÃO — oportunidade de crédito, de 0 a 100

# FÓRMULA  [ALTERADO] (GJ00-60) pesos decididos com o professor em 09/11: mais peso para a capacidade de pagamento
#   O (oportunidade) = (média dos indicadores do Eixo II - 1) / 4      -> de 0 a 1, quanto maior, melhor
#   C (segurança)    = 1 - (média dos indicadores do Eixo I - 1) / 4   -> de 0 a 1, quanto maior, menor o risco
#   score            = 100 x O^0,4 x C^0,6                             -> de 0 a 100
#   Exemplo: média Eixo II = 3,4 e média Eixo I = 2,6  ->  O = 0,6, C = 0,6  ->  score = 60
#   Mercado grande com risco alto (O = 0,9, C = 0,1) -> 24,08 (a média geométrica simples daria 30)
#
# INDICADORES (notas de 1 a 5; 0 nas tabelas dos eixos = SEM DADO e fica fora da média):
#   Eixo I  (risco, maior = pior):        inadimplenciaReal, fragilidadeRenda, agingDivida, vulnerabilidadeSocial
#   Eixo II (oportunidade, maior = melhor): maturidadePix, crescimentoPopulacional, totalHabitantes, bonusDemografico
#
# NÍVEL E FONTE:  [ALTERADO] (GJ00-60) chave UF + sub-região, a mesma da tabela do banco, da API e do site
#   Eixo I  -> calculado POR UF + SUB-REGIÃO a partir do SCR (df_scr_pix), mesma metodologia do calcular_eixo_i,
#              somando as carteiras antes de dividir. Escolarização (IBGE) é a da própria UF.
#   Eixo II -> existe só POR UF (Pix e IBGE): cada sub-região herda as notas da própria UF.
#   Sub-região = UF + 2 primeiros dígitos do CEP. O mesmo código pode existir em UFs diferentes
#              (o 69 existe no AC, AM e RR) e cada par é uma sub-região separada.

import numpy as np
import pandas as pd

INDICADORES_EIXO_I = ["inadimplenciaReal", "fragilidadeRenda", "agingDivida", "vulnerabilidadeSocial"]
INDICADORES_EIXO_II = ["maturidadePix", "crescimentoPopulacional", "totalHabitantes", "bonusDemografico"]
MIN_INDICADORES_POR_EIXO = 1      # mínimo de indicadores com dado em cada eixo para calcular o score

PESO_OPORTUNIDADE = 0.4         # [NOVO] (GJ00-60) expoente de O na fórmula
PESO_CAPACIDADE = 0.6           # [NOVO] (GJ00-60) expoente de C na fórmula

# [ALTERADO] (GJ00-60) Faixas do relatório da Sprint 3 (convenção do time; podem ser ajustadas com o cliente).
# (limite inferior, limite superior, nome). Inferior incluído, superior não (a última faixa inclui o 100).
FAIXAS_SCORE = [
    (0, 30, "Não recomendado"),
    (30, 50, "Baixa atratividade"),
    (50, 70, "Oportunidade com cautela"),
    (70, 100, "Oportunidade prioritária"),
]

SUB_REGIAO_INVALIDA = ["nan", "None", "", "0"]   # o pipeline troca NaN por 0 no final


def score_de_medias(media_eixo_i, media_eixo_ii):
    """Fórmula decidida com o professor. Recebe as médias (notas de 1 a 5) e devolve (O, C, score).
    Aceita número ou Series."""
    o = ((media_eixo_ii - 1) / 4).clip(0, 1) if hasattr(media_eixo_ii, "clip") else min(max((media_eixo_ii - 1) / 4, 0), 1)
    c = (1 - (media_eixo_i - 1) / 4).clip(0, 1) if hasattr(media_eixo_i, "clip") else min(max(1 - (media_eixo_i - 1) / 4, 0), 1)
    return o, c, np.round(100 * o ** PESO_OPORTUNIDADE * c ** PESO_CAPACIDADE, 2)  # [ALTERADO] (GJ00-60)


def _escolarizacao_uf_mes(df_meses_uf, df_ibge):
    """taxa_escolarizacao por (ano_mes, uf), com a mesma regra do calcular_eixo_i (ffill/bfill por UF)."""
    base = df_meses_uf[["ano_mes", "uf"]].drop_duplicates().copy()
    base["ano"] = base["ano_mes"].str[:4]
    ibge = df_ibge[["ano", "uf", "taxa_escolarizacao"]].copy()
    ibge["ano"] = ibge["ano"].astype(str)
    ibge["uf"] = ibge["uf"].astype(str)
    ibge["taxa_escolarizacao"] = ibge["taxa_escolarizacao"].where(ibge["taxa_escolarizacao"] > 0)  # 0 = sem dado
    ibge = ibge.groupby(["ano", "uf"], as_index=False)["taxa_escolarizacao"].mean()
    out = base.merge(ibge, on=["ano", "uf"], how="left").sort_values(["uf", "ano_mes"])
    out["taxa_escolarizacao"] = out.groupby("uf")["taxa_escolarizacao"].transform(lambda x: x.ffill().bfill())
    return out[["ano_mes", "uf", "taxa_escolarizacao"]]


def calcular_eixo_i_por_nivel(df_scr_pix_base, df_ibge_base, nivel=("sub_regiao",)):
    """Eixo I (notas de 1 a 5; 0 = sem dado) mês a mês no nível pedido.
    nivel=("sub_regiao",) é o usado no score; ("regiao", "uf") reproduz o calcular_eixo_i."""
    nivel = list(nivel)
    chave = ["ano_mes"] + nivel

    scr = df_scr_pix_base[
        (df_scr_pix_base["origem"].astype(str) == "scr")
        & (df_scr_pix_base["metrica"].astype(str).isin(["carteira_ativa", "carteira_vencida"]))
    ].copy()
    for c in ["ano_mes", "regiao", "uf", "sub_regiao", "classe", "metrica"]:
        scr[c] = scr[c].astype(str)
    scr["valor"] = pd.to_numeric(scr["valor"], errors="coerce").fillna(0.0)
    if "sub_regiao" in nivel:
        scr = scr[~scr["sub_regiao"].isin(SUB_REGIAO_INVALIDA)]

    ativa = scr[scr["metrica"] == "carteira_ativa"]
    vencida = scr[scr["metrica"] == "carteira_vencida"]
    ativa_dh = ativa[ativa["classe"] == "D-H"]       # mesma regra do calcular_eixo_i (risco alto)

    df = pd.concat(
        [
            ativa.groupby(chave)["valor"].sum().rename("carteira_ativa"),
            vencida.groupby(chave)["valor"].sum().rename("carteira_vencida"),
            ativa_dh.groupby(chave)["valor"].sum().rename("carteira_ativa_dh"),
        ],
        axis=1,
    ).fillna(0.0).reset_index()

    # agingDivida: crescimento da carteira vencida vs. PERIODOS_CRESCIMENTO meses antes
    df["data"] = pd.to_datetime(df["ano_mes"], format="%Y%m")
    passado = df[nivel + ["data", "carteira_vencida"]].rename(columns={"carteira_vencida": "vencida_anterior"})
    passado["data"] = passado["data"] + pd.DateOffset(months=PERIODOS_CRESCIMENTO)
    df = df.merge(passado, on=nivel + ["data"], how="left")
    # Divisão por zero / sem base de comparação = NaN (sem dado). Com 0.0 a normalização trataria como o menor
    # valor e daria nota 1 (risco mínimo): falta de informação virando risco zero.
    df["agingDivida"] = np.where(df["vencida_anterior"] > 0, df["carteira_vencida"] / df["vencida_anterior"] - 1, np.nan)
    df["inadimplenciaReal"] = np.where(df["carteira_ativa"] > 0, df["carteira_vencida"] / df["carteira_ativa"], np.nan)
    df["fragilidadeRenda"] = np.where(df["carteira_ativa"] > 0, df["carteira_ativa_dh"] / df["carteira_ativa"], np.nan)
    df.loc[df["carteira_ativa"] <= 0, "agingDivida"] = np.nan     # sem carteira ativa no mês: não há o que comparar

    # vulnerabilidadeSocial = 1 - escolarização (IBGE, por UF): média das UFs da unidade ponderada pela carteira ativa
    chave_uf = list(dict.fromkeys(chave + ["uf"]))
    pesos = ativa.groupby(chave_uf)["valor"].sum().rename("peso").reset_index()
    pesos = pesos.merge(_escolarizacao_uf_mes(pesos, df_ibge_base), on=["ano_mes", "uf"], how="left")
    pesos = pesos.dropna(subset=["taxa_escolarizacao"])
    pesos["peso_x_taxa"] = pesos["peso"] * pesos["taxa_escolarizacao"]
    esc = pesos.groupby(chave).agg(peso=("peso", "sum"), peso_x_taxa=("peso_x_taxa", "sum")).reset_index()
    esc["taxa_escolarizacao"] = np.where(esc["peso"] > 0, esc["peso_x_taxa"] / esc["peso"], np.nan)
    df = df.merge(esc[chave + ["taxa_escolarizacao"]], on=chave, how="left")
    df["vulnerabilidadeSocial"] = 1 - df["taxa_escolarizacao"]

    for col in INDICADORES_EIXO_I:                   # nota de 1 a 5 (mesma normalizacao do notebook)
        df = normalizacao(df, col)
    df[INDICADORES_EIXO_I] = df[INDICADORES_EIXO_I].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df[chave + INDICADORES_EIXO_I]


def classificar(score, faixas=None):
    """[NOVO] (GJ00-60) Nome da faixa de cada score. Score vazio = "Sem informação"."""
    faixas = FAIXAS_SCORE if faixas is None else faixas
    faixa = pd.Series("Sem informação", index=score.index)
    for i, (inferior, superior, nome) in enumerate(faixas):
        ultima = i == len(faixas) - 1
        dentro = (score >= inferior) & ((score <= superior) if ultima else (score < superior))
        faixa[dentro] = nome
    return faixa


def calcular_score_sub_regiao(df_scr_pix, df_ibge, df_eixo_ii, faixas=None):
    """Score de 0 a 100 por UF + sub-região e mês, com faixa e posição no ranking do mês.
    Devolve os indicadores dos dois eixos junto com médias, O, C, score, faixa e ranking.
    [ALTERADO] (GJ00-60) chave UF + sub-região; o Eixo II vem da própria UF; a coluna de classificação virou "faixa"."""
    eixo_i = calcular_eixo_i_por_nivel(df_scr_pix, df_ibge, ["uf", "sub_regiao"])
    eixo_ii = df_eixo_ii[["ano_mes", "uf", "regiao"] + INDICADORES_EIXO_II].copy()
    for c in ["ano_mes", "uf", "regiao"]:
        eixo_ii[c] = eixo_ii[c].astype(str)
    df = eixo_i.merge(eixo_ii, on=["ano_mes", "uf"], how="left")

    # Médias só com os indicadores que têm informação (0 = sem dado fica fora; nunca vira risco zero)
    for eixo, colunas in (("eixo_i", INDICADORES_EIXO_I), ("eixo_ii", INDICADORES_EIXO_II)):
        com_dado = df[colunas].where(df[colunas] > 0)
        df[f"n_indicadores_{eixo}"] = com_dado.notna().sum(axis=1)
        df[f"media_{eixo}"] = com_dado.mean(axis=1).where(df[f"n_indicadores_{eixo}"] >= MIN_INDICADORES_POR_EIXO)

    df["O"], df["C"], df["score"] = score_de_medias(df["media_eixo_i"], df["media_eixo_ii"])
    df["faixa"] = classificar(df["score"], faixas)
    df["ranking"] = df.groupby("ano_mes")["score"].rank(ascending=False, method="min").astype("Int64")
    df["total_ranqueado"] = df.groupby("ano_mes")["score"].transform("count")

    colunas = (["ano_mes", "uf", "sub_regiao", "regiao"] + INDICADORES_EIXO_I + INDICADORES_EIXO_II +
               ["n_indicadores_eixo_i", "n_indicadores_eixo_ii", "media_eixo_i", "media_eixo_ii",
                "O", "C", "score", "faixa", "ranking", "total_ranqueado"])
    return df[colunas].sort_values(["ano_mes", "ranking", "uf", "sub_regiao"]).reset_index(drop=True)


---
#**EXECUÇÃO PRINCIPAL**
---

In [ ]:
# --- Extração e Pré-Processamento ---
processar()

construir_df_scr_pix()
validar_scr_contra_origem()   # [NOVO]
construir_df_ibge()

df_scr_pix = extract_data('/content/dados/scr_pix.parquet')
df_ibge = extract_data('/content/dados/ibge.parquet')

df_eixo_i = calcular_eixo_i(df_scr_pix, df_ibge)
df_eixo_ii = calcular_eixo_ii(df_scr_pix, df_ibge)
# [NOVO] (GJ00-57) Score por sub-região: Eixo I calculado por sub-região, Eixo II herdado da UF (df_eixo_ii)
df_score_sub_regiao = calcular_score_sub_regiao(df_scr_pix, df_ibge, df_eixo_ii)
# [NOVO] (GJ00-57) Salva o score junto dos indicadores. Fica FORA da limpeza com fillna(0) mais abaixo:
# 0 é um score válido (o pior), então score vazio = sem informação.
salvar_parquet(df_score_sub_regiao, "score_sub_regiao")

df_ibge.drop(columns=["bonusDemografico"], inplace=True)

# --- Limpeza de segurança contra NaNs e infinitos antes do envio ---
df_eixo_i = df_eixo_i.replace([np.inf, -np.inf], np.nan).fillna(0)
df_eixo_ii = df_eixo_ii.replace([np.inf, -np.inf], np.nan).fillna(0)
# [NOVO] "classe" é uma coluna de categorias e fica vazia nas linhas do Pix.
# O fillna(0) não consegue colocar o 0 numa categoria que não existe, então elas viram texto antes
colunas_categoricas = df_scr_pix.select_dtypes("category").columns
df_scr_pix[colunas_categoricas] = df_scr_pix[colunas_categoricas].astype(object)

df_scr_pix = df_scr_pix.replace([np.inf, -np.inf], np.nan).fillna(0)   # [IGUAL]
df_ibge = df_ibge.replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
# [NOVO]Célula inteira nova: exemplo e conferência do score. Rodar DEPOIS da execução principal.
# --- Exemplo e conferência do score (rodar DEPOIS da execução principal) ---
# Exemplo da fórmula: média Eixo II = 3,4 e média Eixo I = 2,6  ->  O = 0,6 e C = 0,6  ->  score = 60
o, c, s = score_de_medias(2.6, 3.4)
assert (round(o, 2), round(c, 2), s) == (0.6, 0.6, 60.0)
print(f"Exemplo: O = {o:.2f} e C = {c:.2f} -> score {s:.0f}")
# [NOVO] (GJ00-60) Caso que separa as fórmulas: mercado grande (O = 0,9) com risco alto (C = 0,1).
# Com mais peso na capacidade de pagamento dá 24,08; a média geométrica simples daria 30.
assert score_de_medias(4.6, 4.6)[2] == 24.08

df = df_score_sub_regiao
# [NOVO] (GJ00-60) a chave é UF + sub-região: nenhum par repetido no mesmo mês
assert not df.duplicated(["ano_mes", "uf", "sub_regiao"]).any(), "UF + sub-região repetida no mesmo mês"
validos = df["score"].dropna()
assert validos.between(0, 100).all(), "há score fora de 0 a 100"

# ranking de cada mês começa em 1 e não passa do total de sub-regiões com score
rk = df.dropna(subset=["score"]).groupby("ano_mes")["ranking"].agg(["min", "max", "count"])
assert (rk["min"] == 1).all() and (rk["max"] <= rk["count"]).all(), "ranking inconsistente"

# falta de informação não vira score: sem indicador em um dos eixos, o score fica vazio
sem_eixo = (df["n_indicadores_eixo_i"] < MIN_INDICADORES_POR_EIXO) | (df["n_indicadores_eixo_ii"] < MIN_INDICADORES_POR_EIXO)
assert df.loc[sem_eixo, "score"].isna().all() and (df.loc[sem_eixo, "faixa"] == "Sem informação").all()

print(f"OK: {len(validos)} scores entre {validos.min():.2f} e {validos.max():.2f} | "
      f"{int(df['score'].isna().sum())} linhas sem informação (score vazio, fora do ranking)")

mes = df.dropna(subset=["score"])["ano_mes"].max()
print(f"\nTop 10 do mês {mes}:")
display(df[df["ano_mes"] == mes].head(10)[["uf", "sub_regiao", "n_indicadores_eixo_i", "n_indicadores_eixo_ii",
                                          "media_eixo_i", "media_eixo_ii", "score", "faixa", "ranking"]])

---
# **Enviar resultados para o banco (GJ00-60)**
---
Grava o score e os indicadores no banco do projeto, pela API Flask (`POST /api/carga`). Rodar só depois da execução principal.

1. No PC com o banco: `flask --app backend/app run` (com `CARGA_TOKEN` preenchido no `backend/.env`).
2. Ligar o túnel para o Flask e copiar o link público que ele mostrar.
3. No Colab, em **Secrets** (ícone da chave, à esquerda), criar `CARGA_TOKEN` com o mesmo valor do `backend/.env`.
4. Colar o link do túnel em `URL_API_LOCAL` abaixo e rodar a célula. Rodar de novo não duplica nada.


In [ ]:
# [NOVO] (GJ00-60) Envia o df_score_sub_regiao para o banco, um mês por vez.
# COMO USAR QUANDO QUISER GRAVAR OS DADOS NO BANCO LOCAL (MariaDB/MySQL):
#   1. No seu PC, configure o arquivo 'backend/.env' definindo um valor para CARGA_TOKEN.
#   2. Inicie a API Flask no terminal: flask --app backend/app run
#   3. Crie um túnel apontando para a porta 5000 (ex.: cloudflared tunnel --url http://localhost:5000)
#      e copie o link HTTPS gerado.
#   4. Aqui no Google Colab, clique no ícone de Chave (Secrets, na barra lateral esquerda),
#      crie um segredo chamado CARGA_TOKEN com o mesmo valor do 'backend/.env' e ative 'Acesso ao notebook'.
#   5. Cole o link do túnel na variável URL_API_LOCAL abaixo e execute esta célula.
#
# OBS.: Se URL_API_LOCAL estiver vazio (""), o envio ao banco é pulado automaticamente
# para que o 'Executar tudo' do Colab não trave e siga direto para a geração dos gráficos!

URL_API_LOCAL = ""  # cole aqui o link do túnel quando quiser gravar no banco (ex.: "https://xxxx.trycloudflare.com")

if URL_API_LOCAL.strip():
    from google.colab import userdata
    enviar_resultados(df_score_sub_regiao, URL_API_LOCAL.strip(), userdata.get("CARGA_TOKEN"))
else:
    print("[Info] Envio ao banco ignorado (URL_API_LOCAL vazio). Os dados continuam no Colab para visualização dos gráficos abaixo.")


---
# **Visualização Gráfica – Eixo I: Risco de Crédito**
---
### Critérios de Aceite Atendidos:
- **Gráficos dos 4 indicadores:** Inadimplência Real, Fragilidade de Renda, Aging da Dívida e Vulnerabilidade Social.
- **Filtros interativos:** Possibilidade de filtrar por período (Ano/Mês) e região (Macrorregião ou Brasil).
- **Formatação completa:** Cada gráfico contém título, eixos nomeados (X e Y) e legenda explicativa.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# (coluna, nome, cor da barra, cor da borda)
INDICADORES_EIXO_I = [
    ("inadimplenciaReal", "Inadimplência Real", "#0284c7", "#0369a1"),
    ("fragilidadeRenda", "Fragilidade de Renda", "#16a34a", "#15803d"),
    ("agingDivida", "Aging da Dívida", "#8b5cf6", "#6d28d9"),
    ("vulnerabilidadeSocial", "Vulnerabilidade Social", "#f59e0b", "#d97706"),
]


def visualizar_indicadores_eixo_i(df_dados, periodo="Todos", regiao="Todas"):
    """
    Gera os 4 gráficos do Eixo I - Risco de Crédito (por estado) com filtros de período e região.
    1. Gráficos de inadimplência real, fragilidade de renda, aging da dívida e vulnerabilidade social
    2. Filtros por período ("Todos", "2024" ou "202412") e região (ou UF)
    3. Título, eixos nomeados e legenda
    Valor 0 no df_eixo_i = "sem dado" (a escala válida é de 1 a 5): fica fora da média do período.
    """
    cols = [c for c, _, _, _ in INDICADORES_EIXO_I]
    df_f = df_dados.copy()
    for c in ["ano_mes", "regiao", "uf"]:
        df_f[c] = df_f[c].astype(str)

    # 1. Filtro por Período
    if periodo and str(periodo).lower() not in ["todos", "todas", "all"]:
        periodo_str = str(periodo)
        if len(periodo_str) == 4:
            df_f = df_f[df_f["ano_mes"].str.startswith(periodo_str)]
        else:
            df_f = df_f[df_f["ano_mes"] == periodo_str]

    # 2. Filtro por Região (ou UF)
    if regiao and str(regiao).lower() not in ["todas", "todos", "brasil", "todas as regiões", "all"]:
        regiao_str = str(regiao).strip()
        if regiao_str in df_f["regiao"].values:
            df_f = df_f[df_f["regiao"] == regiao_str]
        elif regiao_str in df_f["uf"].values:
            df_f = df_f[df_f["uf"] == regiao_str]
        else:
            df_f = df_f.iloc[0:0]

    if df_f.empty:
        print(f"Nenhum registro para Período='{periodo}' e Região='{regiao}'.")
        return None

    # Mês sem dado (valor 0) fica fora da média; a legenda mostra quantos meses entraram
    df_f[cols] = df_f[cols].where(df_f[cols] > 0)
    meses_total = df_f["ano_mes"].nunique()
    meses_com_dado = {c: df_f.loc[df_f[c].notna(), "ano_mes"].nunique() for c in cols}

    df_agg = df_f.groupby(["uf", "regiao"], as_index=False)[cols].mean().sort_values("uf")

    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    fig.patch.set_facecolor("#f8fafc")

    plt.suptitle(
        f"Eixo I – Indicadores de Risco de Crédito\n[Período: {periodo} | Região: {regiao}]",
        fontsize=16, fontweight="bold", color="#0f172a", y=0.98
    )

    for i, (ax, (col, nome, cor, borda)) in enumerate(zip(axes.flat, INDICADORES_EIXO_I), start=1):
        ax.set_facecolor("#ffffff")
        ax.set_title(f"{i}. {nome} por Estado", pad=10, fontweight="bold")
        ax.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
        ax.set_ylabel(f"Índice de {nome} (Escala 1 a 5)", fontweight="bold")
        ax.grid(True, linestyle=":", alpha=0.6)

        dados = df_agg[["uf", col]].dropna()
        if dados.empty:
            ax.text(0.5, 0.5, "Sem dados neste período", ha="center", va="center",
                    transform=ax.transAxes, fontsize=12, color="gray")
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        meses_txt = f" – {meses_com_dado[col]}/{meses_total} meses" if meses_com_dado[col] < meses_total else ""
        barras = ax.bar(dados["uf"], dados[col], color=cor, alpha=0.85, edgecolor=borda,
                        label=f"{nome} (Normalizado 1-5){meses_txt}")
        media = dados[col].mean()
        ax.axhline(media, color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({media:.2f})")
        ax.set_ylim(0, 5.5)
        if len(dados) < 3:  # poucas barras (ex.: uma UF só) não devem ocupar o gráfico inteiro
            ax.set_xlim(-1.5, len(dados) + 0.5)
        ax.legend(loc="upper right", frameon=True)
        for b in barras:
            h = b.get_height()
            ax.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8, fontweight="bold")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


# Abre os filtros (dropdowns) no topo, como no Eixo II
if "df_eixo_i" in globals():
    meses = sorted(df_eixo_i["ano_mes"].astype(str).unique(), reverse=True)
    anos = sorted({m[:4] for m in meses}, reverse=True)
    interact(
        lambda periodo, regiao: visualizar_indicadores_eixo_i(df_eixo_i, periodo, regiao),
        periodo=widgets.Dropdown(options=["Todos"] + anos + meses, value="Todos", description="Período:"),
        regiao=widgets.Dropdown(options=["Todas"] + sorted(df_eixo_i["regiao"].astype(str).unique()),
                                value="Todas", description="Região:"),
    )
else:
    print("Rode a EXECUÇÃO PRINCIPAL (gera o df_eixo_i) e depois esta célula de novo.")

---
# **Visualização Gráfica – Eixo II: Inclusão e Expansão Demográfica**
---
### Critérios de Aceite Atendidos:
- **Gráficos dos 4 indicadores:** Maturidade Pix, Crescimento Populacional, Total de Habitantes e Bônus Demográfico.
- **Filtros interativos:** Possibilidade de filtrar por período (Ano/Mês) e região (Macrorregião ou Brasil).
- **Formatação completa:** Cada gráfico contém título, eixos nomeados (X e Y) e legenda explicativa.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def visualizar_indicadores_eixo_ii(df_dados, periodo="Todos", regiao="Todas", salvar_caminho=None):
    """
    Gera os 4 gráficos do Eixo II - Inclusão e Demografia com filtros de período e região.
    Atende integralmente aos critérios de aceite:
    1. Gráficos de maturidade Pix, crescimento populacional, total de habitantes e bônus demográfico
    2. Filtros por período e região
    3. Título, eixos nomeados e legenda
    """
    df_f = df_dados.copy()
    
    # 1. Filtro por Período
    if periodo and str(periodo).lower() not in ["todos", "todas", "all"]:
        periodo_str = str(periodo)
        if len(periodo_str) == 4:
            df_f = df_f[df_f["ano_mes"].astype(str).str.startswith(periodo_str)]
        else:
            df_f = df_f[df_f["ano_mes"].astype(str) == periodo_str]
            
    # 2. Filtro por Região
    if regiao and str(regiao).lower() not in ["todas", "todos", "brasil", "todas as regiões", "all"]:
        regiao_str = str(regiao).strip()
        if regiao_str in df_f["regiao"].values:
            df_f = df_f[df_f["regiao"] == regiao_str]
        elif regiao_str in df_f["uf"].values:
            df_f = df_f[df_f["uf"] == regiao_str]
            
    if df_f.empty:
        print(f"Nenhum registro para Período='{periodo}' e Região='{regiao}'.")
        return None
        
    df_agg = df_f.groupby(["uf", "regiao"], as_index=False).agg({
        "maturidadePix": "mean",
        "crescimentoPopulacional": "mean",
        "totalHabitantes": "mean",
        "bonusDemografico": "mean"
    }).sort_values("uf")
    
    ufs = df_agg["uf"].tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    fig.patch.set_facecolor("#f8fafc")
    
    plt.suptitle(
        f"Eixo II – Indicadores de Inclusão e Expansão Demográfica\n[Período: {periodo} | Região: {regiao}]",
        fontsize=16, fontweight="bold", color="#0f172a", y=0.98
    )
    
    # 1. Maturidade Pix
    ax1 = axes[0, 0]
    ax1.set_facecolor("#ffffff")
    b1 = ax1.bar(ufs, df_agg["maturidadePix"], color="#0284c7", alpha=0.85, edgecolor="#0369a1", label="Maturidade Pix (Normalizado 1-5)")
    ax1.axhline(df_agg["maturidadePix"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['maturidadePix'].mean():.2f})")
    ax1.set_title("1. Maturidade Pix por Estado", pad=10, fontweight="bold")
    ax1.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax1.set_ylabel("Índice de Maturidade Pix (Escala 1 a 5)", fontweight="bold")
    ax1.set_ylim(0, 5.5)
    ax1.legend(loc="upper right", frameon=True)
    ax1.grid(True, linestyle=":", alpha=0.6)
    for b in b1:
        h = b.get_height()
        ax1.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 2. Crescimento Populacional
    ax2 = axes[0, 1]
    ax2.set_facecolor("#ffffff")
    b2 = ax2.bar(ufs, df_agg["crescimentoPopulacional"], color="#16a34a", alpha=0.85, edgecolor="#15803d", label="Crescimento Populacional (Normalizado 1-5)")
    ax2.axhline(df_agg["crescimentoPopulacional"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['crescimentoPopulacional'].mean():.2f})")
    ax2.set_title("2. Crescimento Populacional por Estado (Censo IBGE)", pad=10, fontweight="bold")
    ax2.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax2.set_ylabel("Índice de Crescimento Populacional (Escala 1 a 5)", fontweight="bold")
    ax2.set_ylim(0, 5.5)
    ax2.legend(loc="upper right", frameon=True)
    ax2.grid(True, linestyle=":", alpha=0.6)
    for b in b2:
        h = b.get_height()
        ax2.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 3. Total de Habitantes
    ax3 = axes[1, 0]
    ax3.set_facecolor("#ffffff")
    b3 = ax3.bar(ufs, df_agg["totalHabitantes"], color="#8b5cf6", alpha=0.85, edgecolor="#6d28d9", label="Total de Habitantes (Normalizado 1-5)")
    ax3.axhline(df_agg["totalHabitantes"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['totalHabitantes'].mean():.2f})")
    ax3.set_title("3. Total de Habitantes por Estado (População Residente)", pad=10, fontweight="bold")
    ax3.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax3.set_ylabel("Índice de Total de Habitantes (Escala 1 a 5)", fontweight="bold")
    ax3.set_ylim(0, 5.5)
    ax3.legend(loc="upper right", frameon=True)
    ax3.grid(True, linestyle=":", alpha=0.6)
    for b in b3:
        h = b.get_height()
        ax3.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 4. Bônus Demográfico
    ax4 = axes[1, 1]
    ax4.set_facecolor("#ffffff")
    b4 = ax4.bar(ufs, df_agg["bonusDemografico"], color="#f59e0b", alpha=0.85, edgecolor="#d97706", label="Bônus Demográfico (Normalizado 1-5)")
    ax4.axhline(df_agg["bonusDemografico"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['bonusDemografico'].mean():.2f})")
    ax4.set_title("4. Bônus Demográfico por Estado (Janela Etária)", pad=10, fontweight="bold")
    ax4.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax4.set_ylabel("Índice de Bônus Demográfico (Escala 1 a 5)", fontweight="bold")
    ax4.set_ylim(0, 5.5)
    ax4.legend(loc="upper right", frameon=True)
    ax4.grid(True, linestyle=":", alpha=0.6)
    for b in b4:
        h = b.get_height()
        ax4.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    if salvar_caminho:
        plt.savefig(salvar_caminho, dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Filtros Interativos por Período e Região para o Google Colab
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    # Obter opções disponíveis a partir do df_eixo_ii
    periodos_disponiveis = ["Todos"] + sorted(df_eixo_ii["ano_mes"].astype(str).unique().tolist())
    regioes_disponiveis = ["Todas"] + sorted(df_eixo_ii["regiao"].dropna().unique().tolist())
    
    dropdown_periodo = widgets.Dropdown(
        options=periodos_disponiveis,
        value="Todos",
        description="Período:",
        style={"description_width": "initial"}
    )
    
    dropdown_regiao = widgets.Dropdown(
        options=regioes_disponiveis,
        value="Todas",
        description="Região:",
        style={"description_width": "initial"}
    )
    
    def atualizar(periodo, regiao):
        visualizar_indicadores_eixo_ii(df_eixo_ii, periodo=periodo, regiao=regiao)
        
    display(widgets.interactive(atualizar, periodo=dropdown_periodo, regiao=dropdown_regiao))
except Exception as e:
    print(f"Executando visualização estática (ipywidgets opcional): {e}")
    visualizar_indicadores_eixo_ii(df_eixo_ii, periodo="Todos", regiao="Todas")


In [ ]:
# Exemplo: Visualização dos gráficos filtrando por Região="Nordeste" e Período mais recente
visualizar_indicadores_eixo_ii(df_eixo_ii, periodo="Todos", regiao="Nordeste")


---
# **Visualização Gráfica – Evolução do Score de Oportunidade (GJ00-70)**
---
### Critérios de Aceite Atendidos:
- **Gráfico de linhas da evolução mensal:** Mostra a trajetória do **Score de Oportunidade (0 a 100)** de uma sub-região (`UF + sub_regiao`) ao longo dos meses disponíveis, junto com a média da própria sub-região no período, a média mensal de referência (regional/estadual/nacional) e as faixas oficiais de classificação (`FAIXAS_SCORE`).
- **Filtros por período, região/UF e sub-região:** Permite escolher a **Região / UF**, a **Sub-região (`UF-código`)** e o **Período** (`Todos` ou ano específico) conforme os dados disponíveis no notebook.
- **Formatação completa e explicação simples:** Inclui título, nomes dos eixos X e Y, legenda clara e guia rápido de leitura logo abaixo.
- **Reuso dos cálculos do Colab:** Utiliza diretamente o `df_score_sub_regiao` já calculado (sem refazer o cálculo do score e sem alterar a API ou o site).

---
### Como Ler o Gráfico de Evolução do Score (Explicação Simples)
- **Eixo Horizontal (X — Período):** Mostra os meses (`AAAA-MM`) em ordem cronológica.
- **Eixo Vertical (Y — Score de Oportunidade, de 0 a 100):** Mostra a nota alcançada pela sub-região (`UF-código`) em cada mês.
- **Linha Azul Contínua (com pontos):** É a evolução do Score da sub-região escolhida.
  - Se a linha **sobe**, a sub-região ficou **mais atrativa para concessão de crédito** (melhora no potencial demográfico/Pix e/ou queda no risco de crédito).
  - Se a linha **desce**, houve perda de oportunidade ou aumento do risco no período.
- **Linha Tracejada Vermelha:** Indica a **média histórica** da própria sub-região no período selecionado.
- **Linha Pontilhada Cinza:** Mostra a **média das sub-regiões da mesma região/UF/país** em cada mês, permitindo comparar se a sub-região está acima ou abaixo dos seus pares.
- **Faixas Coloridas no Fundo (`FAIXAS_SCORE`):** Indicam a classificação oficial do Score:
  - `0 a 30`: **Não recomendado**
  - `30 a 50`: **Baixa atratividade**
  - `50 a 70`: **Oportunidade com cautela**
  - `70 a 100`: **Oportunidade prioritária**


In [ ]:
# [NOVO] (GJ00-70) Funções para o Gráfico de Evolução Mensal do Score de Oportunidade por Sub-região (UF + sub_regiao)
import traceback
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# Paleta alinhada às 4 faixas de FAIXAS_SCORE definidas na célula de cálculo do Score:
# (0, 30, "Não recomendado"), (30, 50, "Baixa atratividade"), (50, 70, "Oportunidade com cautela"), (70, 100, "Oportunidade prioritária")
ESTILO_FAIXAS_PADRAO = [
    ("#ef4444", "#b91c1c", "#fef2f2"),  # 0 a 30: Não recomendado
    ("#f59e0b", "#b45309", "#fffbeb"),  # 30 a 50: Baixa atratividade
    ("#0284c7", "#0369a1", "#f0f9ff"),  # 50 a 70: Oportunidade com cautela
    ("#16a34a", "#15803d", "#f0fdf4"),  # 70 a 100: Oportunidade prioritária
]


def _obter_faixas_score():
    """Retorna a lista FAIXAS_SCORE definida no notebook, garantindo consistência com o banco, a API e o site."""
    if "FAIXAS_SCORE" in globals() and isinstance(globals()["FAIXAS_SCORE"], list):
        return globals()["FAIXAS_SCORE"]
    return [
        (0, 30, "Não recomendado"),
        (30, 50, "Baixa atratividade"),
        (50, 70, "Oportunidade com cautela"),
        (70, 100, "Oportunidade prioritária"),
    ]


def carregar_dados_score_existentes(df_dados=None):
    """
    Obtém o DataFrame df_score_sub_regiao já calculado no Colab (em memória ou salvo em parquet),
    sem refazer o cálculo do score nem alterar a API ou o site.
    """
    # 1. Se o DataFrame já foi passado diretamente por parâmetro, usa uma cópia dele
    if df_dados is not None:
        return df_dados.copy()
    # 2. Se já existe na memória do Colab (gerado na seção EXECUÇÃO PRINCIPAL), reutiliza direto
    if "df_score_sub_regiao" in globals() and isinstance(globals()["df_score_sub_regiao"], pd.DataFrame):
        return globals()["df_score_sub_regiao"].copy()
    # 3. Caso a sessão tenha sido reiniciada, tenta ler o parquet já salvo na pasta /content/dados
    caminho_parquet = Path("/content/dados/score_sub_regiao.parquet")
    if caminho_parquet.exists():
        return pd.read_parquet(caminho_parquet)
    raise RuntimeError(
        "O DataFrame 'df_score_sub_regiao' ainda não está disponível. "
        "Execute a seção 'EXECUÇÃO PRINCIPAL' antes de gerar o gráfico."
    )


def _formatar_ano_mes(ano_mes_str):
    """Converte o formato AAAAMM (ex.: '202412') para AAAA-MM ('2024-12') para facilitar a leitura no eixo X."""
    s = str(ano_mes_str).strip()
    if len(s) == 6 and s.isdigit():
        return f"{s[:4]}-{s[4:]}"
    return s


def _filtrar_por_regiao_ou_uf(df, regiao):
    """Filtra o DataFrame de score por macrorregião ('Sudeste', 'Nordeste', etc.) ou por sigla de UF ('SP', 'MG', etc.)."""
    if not regiao or str(regiao).strip().lower() in ["todas", "todos", "brasil", "todas as regiões", "all"]:
        return df
    reg_str = str(regiao).strip()
    if "regiao" in df.columns and reg_str in df["regiao"].astype(str).values:
        return df[df["regiao"].astype(str) == reg_str]
    if "uf" in df.columns and reg_str in df["uf"].astype(str).values:
        return df[df["uf"].astype(str) == reg_str]
    return df.iloc[0:0]


def _filtrar_por_periodo(df, periodo):
    """Filtra o DataFrame pelo período escolhido ('Todos', ano '2024', mês '202412' ou intervalo '202401-202412')."""
    if not periodo or str(periodo).strip().lower() in ["todos", "todas", "all"]:
        return df
    p_str = str(periodo).strip().replace("-", "")
    if len(p_str) == 4 and p_str.isdigit():
        return df[df["ano_mes"].astype(str).str.startswith(p_str)]
    if len(p_str) == 6 and p_str.isdigit():
        return df[df["ano_mes"].astype(str) == p_str]
    if len(p_str) == 12 and p_str.isdigit():
        ini, fim = p_str[:6], p_str[6:]
        return df[(df["ano_mes"].astype(str) >= ini) & (df["ano_mes"].astype(str) <= fim)]
    return df


def _interpretar_chave_sub_regiao(sub_regiao_param, df_escopo):
    """
    Resolve o par (uf, sub_regiao) a partir do parâmetro informado pelo usuário ou dropdown.
    Aceita formatos como 'SP-01 (Sudeste)', 'SP-01', 'SP|01', tupla ('SP', '01') ou código '01'.
    Como o mesmo código de sub_regiao pode existir em UFs diferentes (ex.: 69 em AC, AM e RR),
    sempre retorna o par (uf, sub_regiao) exato.
    """
    validos = df_escopo.dropna(subset=["score"])
    base_busca = validos if not validos.empty else df_escopo
    if base_busca.empty:
        return None, None

    if isinstance(sub_regiao_param, (tuple, list)) and len(sub_regiao_param) == 2:
        return str(sub_regiao_param[0]).strip(), str(sub_regiao_param[1]).strip()

    texto = str(sub_regiao_param or "").strip()
    if not texto or texto.lower().startswith("autom"):
        melhor = (
            base_busca.groupby(["uf", "sub_regiao"])["score"]
            .mean()
            .sort_values(ascending=False)
            .index[0]
        )
        return str(melhor[0]), str(melhor[1])

    # Remove sufixo descritivo do dropdown, ex.: "SP-01 (Sudeste)" -> "SP-01"
    if " (" in texto:
        texto = texto.split(" (", 1)[0].strip()

    for sep in ("-", "|", "/"):
        if sep in texto:
            partes = [p.strip() for p in texto.split(sep, 1)]
            if len(partes) == 2 and partes[0] and partes[1]:
                return partes[0], partes[1]

    # Se informou apenas o código da sub-região (ex.: "01"), escolhe dentro do escopo filtrado a UF de maior score médio
    candidatos = base_busca[base_busca["sub_regiao"] == texto]
    if not candidatos.empty:
        uf_escolhida = (
            candidatos.groupby("uf")["score"]
            .mean()
            .sort_values(ascending=False)
            .index[0]
        )
        return str(uf_escolhida), texto

    return None, texto


def visualizar_evolucao_score(
    df_dados=None,
    sub_regiao="Automática (Melhor Score)",
    periodo="Todos",
    regiao="Todas",
    salvar_caminho=None,
    ax=None
):
    """
    Gera o gráfico de linhas mostrando a evolução mensal do Score de Oportunidade (0 a 100)
    de uma sub-região (UF + sub_regiao) ao longo dos meses disponíveis.
    """
    # 1. Carrega os resultados já calculados no Colab (sem recalcular o score)
    df = carregar_dados_score_existentes(df_dados)
    for col in ["ano_mes", "uf", "sub_regiao", "regiao", "faixa"]:
        if col in df.columns:
            df[col] = df[col].astype(str)

    # 2. Aplica os filtros de período e região/UF escolhidos pelo analista
    df_periodo = _filtrar_por_periodo(df, periodo)
    df_contexto = _filtrar_por_regiao_ou_uf(df_periodo, regiao)

    criou_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(13, 6.5))
        fig.patch.set_facecolor("#f8fafc")
        criou_fig = True

    ax.set_facecolor("#ffffff")

    if df_contexto.empty:
        msg = f"Nenhum registro encontrado para Período='{periodo}' e Região/UF='{regiao}'."
        print(msg)
        ax.text(0.5, 0.5, msg, ha="center", va="center", transform=ax.transAxes, fontsize=11, color="#64748b")
        ax.set_title("Evolução Mensal do Score de Oportunidade", pad=10, fontweight="bold")
        ax.set_xlabel("Período de Referência (Ano-Mês)", fontweight="bold")
        ax.set_ylabel("Score de Oportunidade (Escala 0 a 100)", fontweight="bold")
        if criou_fig:
            plt.tight_layout()
            plt.show()
        return None

    # 3. Resolve o par (uf, sub_regiao) dentro do contexto filtrado
    uf_sub, cod_sub = _interpretar_chave_sub_regiao(sub_regiao, df_contexto)
    # Caso o usuário tenha passado explicitamente uma UF-sub_regiao de outra região, busca no df_periodo
    if uf_sub is not None:
        df_sub = df_contexto[(df_contexto["uf"] == uf_sub) & (df_contexto["sub_regiao"] == cod_sub)].sort_values("ano_mes").copy()
        if df_sub.empty:
            df_sub = df_periodo[(df_periodo["uf"] == uf_sub) & (df_periodo["sub_regiao"] == cod_sub)].sort_values("ano_mes").copy()
    else:
        df_sub = df_contexto[df_contexto["sub_regiao"] == str(cod_sub)].sort_values("ano_mes").copy()

    df_sub_valido = df_sub.dropna(subset=["score"]).copy()
    chave_exibicao = f"{uf_sub}-{cod_sub}" if uf_sub else str(cod_sub)

    # Tratamento caso não existam meses com score válido para a sub-região selecionada
    if df_sub_valido.empty:
        ax.text(
            0.5, 0.5,
            f"Sem dados válidos de Score para a sub-região '{chave_exibicao}' no período '{periodo}'.",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="#64748b"
        )
        ax.set_title(f"Evolução Mensal do Score de Oportunidade – Sub-região {chave_exibicao}", pad=10, fontweight="bold")
        ax.set_xlabel("Período de Referência (Ano-Mês)", fontweight="bold")
        ax.set_ylabel("Score de Oportunidade (Escala 0 a 100)", fontweight="bold")
        if criou_fig:
            plt.tight_layout()
            plt.show()
        return chave_exibicao

    uf_sub = df_sub_valido["uf"].iloc[-1]
    cod_sub = df_sub_valido["sub_regiao"].iloc[-1]
    reg_sub = df_sub_valido["regiao"].iloc[-1] if "regiao" in df_sub_valido.columns else "-"
    chave_exibicao = f"{uf_sub}-{cod_sub}"

    # 4. Desenha as faixas horizontais de fundo usando FAIXAS_SCORE oficial (0-30, 30-50, 50-70, 70-100)
    faixas_oficiais = _obter_faixas_score()
    for idx_f, (inf, sup, nome_faixa) in enumerate(faixas_oficiais):
        _, _, cor_fundo = ESTILO_FAIXAS_PADRAO[idx_f % len(ESTILO_FAIXAS_PADRAO)]
        ax.axhspan(inf, sup, color=cor_fundo, alpha=0.55, zorder=0, label=f"{nome_faixa} ({inf}–{sup})")
        if inf > 0:
            ax.axhline(inf, color="#cbd5e1", linestyle=":", linewidth=0.9, zorder=1)

    meses_raw = df_sub_valido["ano_mes"].tolist()
    meses_fmt = [_formatar_ano_mes(m) for m in meses_raw]
    x_idx = np.arange(len(meses_fmt))
    scores_sub = df_sub_valido["score"].values

    # 5. Plota a linha pontilhada da média mensal da Região/UF/Brasil para servir de referência comparativa
    validos_ctx = df_contexto.dropna(subset=["score"])
    if not validos_ctx.empty:
        media_mensal_ctx = validos_ctx.groupby("ano_mes")["score"].mean().reindex(meses_raw)
        rotulo_ctx = f"Média {regiao}" if str(regiao).strip().lower() not in ["todas", "todos", "brasil", "all"] else "Média Brasil"
        ax.plot(
            x_idx, media_mensal_ctx.values,
            color="#64748b", linestyle=":", marker="s", markersize=4.5, linewidth=1.7, alpha=0.85,
            label=f"{rotulo_ctx} no mês", zorder=2
        )

    # 6. Plota a linha principal (azul) com a evolução mensal do Score da sub-região (UF + sub_regiao)
    ax.plot(
        x_idx, scores_sub,
        color="#0284c7", linestyle="-", marker="o", markersize=7.5, linewidth=2.6,
        markerfacecolor="#0284c7", markeredgecolor="#ffffff", markeredgewidth=1.5,
        label=f"Score Sub-região {chave_exibicao} ({reg_sub})", zorder=4
    )

    # 7. Plota a linha tracejada vermelha com a média da própria sub-região no período
    media_sub = float(np.mean(scores_sub))
    ax.axhline(
        media_sub, color="#dc2626", linestyle="--", linewidth=1.6,
        label=f"Média da Sub-região ({media_sub:.1f} pts)", zorder=3
    )

    # 8. Anota os valores numéricos do Score sobre os pontos (espaçando caso haja muitos meses)
    passo_anotacao = max(1, len(x_idx) // 12)
    for i, (x_val, y_val) in enumerate(zip(x_idx, scores_sub)):
        if i % passo_anotacao == 0 or i == len(x_idx) - 1 or y_val in (scores_sub.min(), scores_sub.max()):
            ax.annotate(
                f"{y_val:.1f}",
                (x_val, y_val),
                xytext=(0, 8), textcoords="offset points",
                ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#0f172a",
                bbox=dict(boxstyle="round,pad=0.18", facecolor="#ffffff", edgecolor="#bae6fd", alpha=0.88)
            )

    # 9. Configura títulos, nomes dos eixos X e Y, grade e legenda
    passo_ticks = max(1, len(x_idx) // 14)
    ax.set_xticks(x_idx[::passo_ticks])
    ax.set_xticklabels([meses_fmt[i] for i in range(0, len(meses_fmt), passo_ticks)], rotation=45, ha="right")
    if len(x_idx) == 1:
        ax.set_xlim(-0.5, 0.5)

    ax.set_ylim(0, 105)
    ax.set_title(
        f"Evolução Mensal do Score de Oportunidade – Sub-região {chave_exibicao} ({reg_sub})\n"
        f"[Período: {periodo} | Filtro de Referência: {regiao}]",
        pad=12, fontsize=13.5, fontweight="bold", color="#0f172a"
    )
    ax.set_xlabel("Período de Referência (Ano-Mês)", fontweight="bold", labelpad=8)
    ax.set_ylabel("Score de Oportunidade (Escala 0 a 100)", fontweight="bold", labelpad=8)
    ax.grid(True, linestyle=":", alpha=0.6, zorder=1)
    ax.legend(loc="upper left", ncol=2, frameon=True, facecolor="#ffffff", edgecolor="#e2e8f0", fontsize=8.5)

    if criou_fig:
        plt.tight_layout()
        if salvar_caminho:
            plt.savefig(salvar_caminho, dpi=200, bbox_inches="tight")
        plt.show()
        faixa_atual = df_sub_valido["faixa"].iloc[-1] if "faixa" in df_sub_valido.columns else "-"
        print(
            f"COMO LER ESTE GRÁFICO (Sub-região {chave_exibicao} – {reg_sub} | Faixa no último mês: {faixa_atual}):\n"
            f" • A linha azul mostra a evolução mensal do Score (mínimo: {scores_sub.min():.1f} | máximo: {scores_sub.max():.1f} | média: {media_sub:.1f}).\n"
            " • Quando a linha sobe, a sub-região ganhou atratividade de crédito; quando desce, perdeu potencial ou teve aumento de risco.\n"
            " • Compare a linha azul com a linha pontilhada cinza para ver em quais meses a sub-região ficou acima da média de referência."
        )

    return chave_exibicao


In [ ]:
# [NOVO] (GJ00-70) Filtros Interativos para o Gráfico de Evolução do Score no Google Colab
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as e:
    print(f"ipywidgets não disponível ({e}); exibindo visualização estática:")
    if "df_score_sub_regiao" in globals():
        visualizar_evolucao_score(df_score_sub_regiao, periodo="Todos", regiao="Todas")
else:
    if "df_score_sub_regiao" not in globals() and not Path("/content/dados/score_sub_regiao.parquet").exists():
        print("Rode a EXECUÇÃO PRINCIPAL (gera o df_score_sub_regiao) e depois execute esta célula novamente.")
    else:
        try:
            # Lê os dados já calculados de df_score_sub_regiao para montar as listas de opções dos filtros
            df_score_ref = carregar_dados_score_existentes()
            df_score_val = df_score_ref.dropna(subset=["score"]).copy()
            for c in ["ano_mes", "uf", "sub_regiao", "regiao"]:
                df_score_val[c] = df_score_val[c].astype(str)

            meses_disponiveis = sorted(df_score_val["ano_mes"].unique().tolist(), reverse=True)
            anos_disponiveis = sorted({m[:4] for m in meses_disponiveis}, reverse=True)
            regioes_macro = sorted(df_score_val["regiao"].dropna().unique().tolist())
            ufs_disponiveis = sorted(df_score_val["uf"].dropna().unique().tolist())

            # Monta a lista de pares UF-sub_regiao disponíveis de acordo com a Região/UF selecionada
            def _opcoes_sub_regiao_evol(regiao_sel):
                df_f = _filtrar_por_regiao_ou_uf(df_score_val, regiao_sel)
                if df_f.empty:
                    df_f = df_score_val
                resumo = (
                    df_f.groupby(["uf", "sub_regiao", "regiao"], as_index=False)["score"]
                    .mean()
                    .sort_values(["uf", "sub_regiao"])
                )
                opcoes = ["Automática (Melhor Score)"]
                for _, r in resumo.iterrows():
                    opcoes.append(f"{r['uf']}-{r['sub_regiao']} ({r['regiao']})")
                return opcoes

            # Dropdown 1: Escolha da Região ou UF
            dropdown_regiao_evol = widgets.Dropdown(
                options=["Todas"] + regioes_macro + ufs_disponiveis,
                value="Todas",
                description="Região / UF:",
                style={"description_width": "initial"},
                layout=widgets.Layout(width="240px")
            )

            # Dropdown 2: Escolha da Sub-região (chave UF-sub_regiao)
            dropdown_sub_evol = widgets.Dropdown(
                options=_opcoes_sub_regiao_evol("Todas"),
                value="Automática (Melhor Score)",
                description="Sub-região:",
                style={"description_width": "initial"},
                layout=widgets.Layout(width="300px")
            )

            # Dropdown 3: Escolha do Período (Todos os meses ou um ano específico)
            dropdown_periodo_evol = widgets.Dropdown(
                options=["Todos"] + anos_disponiveis,
                value="Todos",
                description="Período:",
                style={"description_width": "initial"},
                layout=widgets.Layout(width="220px")
            )

            # Atualiza automaticamente as sub-regiões disponíveis quando o analista troca a Região/UF
            def _ao_mudar_regiao_evol(change):
                dropdown_sub_evol.options = _opcoes_sub_regiao_evol(change["new"])
                dropdown_sub_evol.value = "Automática (Melhor Score)"

            dropdown_regiao_evol.observe(_ao_mudar_regiao_evol, names="value")

            saida_evolucao = widgets.Output()

            # Função que redesenha o gráfico sempre que qualquer filtro é alterado
            def _atualizar_evolucao(*args):
                with saida_evolucao:
                    saida_evolucao.clear_output(wait=True)
                    visualizar_evolucao_score(
                        df_score_ref,
                        sub_regiao=dropdown_sub_evol.value,
                        periodo=dropdown_periodo_evol.value,
                        regiao=dropdown_regiao_evol.value,
                    )

            for w in (dropdown_regiao_evol, dropdown_sub_evol, dropdown_periodo_evol):
                w.observe(_atualizar_evolucao, names="value")

            display(widgets.HBox([dropdown_regiao_evol, dropdown_sub_evol, dropdown_periodo_evol]), saida_evolucao)
            _atualizar_evolucao()
        except Exception:
            traceback.print_exc()


In [ ]:
# [NOVO] (GJ00-70) Exemplo de chamada direta: evolução do Score para a melhor sub-região do Sudeste em todo o período
if "df_score_sub_regiao" in globals():
    visualizar_evolucao_score(
        df_score_sub_regiao,
        sub_regiao="Automática (Melhor Score)",
        periodo="Todos",
        regiao="Sudeste"
    )
else:
    print("Rode a EXECUÇÃO PRINCIPAL (gera o df_score_sub_regiao) e depois execute esta célula.")
